# Assignment 1.3

Let's implement h1 heuristic from the slides : 

In [1]:
def h1(self, state):
        """h1: sum of Manhattan distances of tiles to their goal positions as in the slides"""
        N = self.size
        h = 0
        for i, tile in enumerate(state):
            if tile == SPACE:
                continue
            j = self.goal_pos[tile]
            h += abs(i // N - j // N) + abs(i % N - j % N)
        return h

In [2]:
import time
from state_search_thin import StateSearch
from slide_domain_thin import SlideDomain, problems, path_solved, path_compare, SPACE

SlideDomain.h1 = h1
SlideDomain.astar_cost = lambda self, path: len(path) // 2 + self.h1(path[-1])


def run_strategy (strategy, model):
    """ Run a particular search problem. """
    tick = time.perf_counter()
    result = strategy(model)
    tock = time.perf_counter()

    # Output result
    elap = tock - tick
    print (f'Time: {elap:6.2f} s using {strategy.__name__.upper()}')
    solution = path_solved(result)
    print (f'Found {len(solution)} step solution:')
    print (solution)
    return result

for k in range(1, 8):
    problem = problems[k]
    print (f"\n========== problems[{k}] (stored answer: {len(problem.answer)} steps) ==========\n")

    model = (problem.start, problem.is_goal, problem.go_next, problem.astar_cost)
    so = StateSearch(len(problem.answer))

    result = run_strategy (so.lcgq, model)

    solution = path_solved(result)
    if 0 in problem.answer:
        print (f'No stored answer (only an upper bound of {len(problem.answer)} steps)')
    elif path_compare(result, problem.answer):
        print ('Same solution as the stored answer')
    else:
        print (f'Different solution from the stored answer ({len(solution)} vs {len(problem.answer)} steps)')
        print (problem.answer)


========== problems[1] (stored answer: 9 steps) ==========

Time:   0.00 s using LCGQ
Found 9 step solution:
['L', 'U', 'R', 'R', 'R', 'U', 'L', 'L', 'L']
Same solution as the stored answer

========== problems[2] (stored answer: 15 steps) ==========

Time:   0.00 s using LCGQ
Found 15 step solution:
['R', 'R', 'R', 'U', 'L', 'L', 'L', 'U', 'R', 'R', 'R', 'U', 'L', 'L', 'L']
Same solution as the stored answer

========== problems[3] (stored answer: 20 steps) ==========

Time:   0.00 s using LCGQ
Found 20 step solution:
['L', 'L', 'D', 'D', 'D', 'R', 'R', 'R', 'U', 'L', 'L', 'L', 'U', 'R', 'R', 'R', 'U', 'L', 'L', 'L']
Same solution as the stored answer

========== problems[4] (stored answer: 22 steps) ==========

Time:   0.00 s using LCGQ
Found 22 step solution:
['U', 'L', 'L', 'L', 'D', 'D', 'D', 'R', 'R', 'R', 'U', 'L', 'L', 'L', 'U', 'R', 'R', 'R', 'U', 'L', 'L', 'L']
Same solution as the stored answer

========== problems[5] (stored answer: 30 steps) ==========

Time:   0.05 s usi

This heuristic doesn't work with problem 8, let's understand why.

Recall from first assigment :

```python
def graphsearch2 (model, maxdepth=10, strategy="BFS"):
    (start, goal, successors) = model
    cNewPath, cNewStates = {}, {}
    deja_vu = [start]
    if goal(start): return ([start], {}, {})
    frontier = [ [start] ]
    while frontier:
        path = frontier.pop() if strategy == "DFS" else frontier.pop(0) # Strategy selection
        if len(path)//2 >= maxdepth: continue
        state = path[-1]
        l = (len(path)) // 2 + 1
        for (action, new_state) in successors(state):
            cNewStates[l] = cNewStates.get(l, 0) + 1
            if new_state not in deja_vu :
                deja_vu.append(new_state)
                new_path = path + [action, new_state]
                cNewPath[l] = cNewPath.get(l, 0) + 1
                if goal(new_state): 
                    return (new_path, cNewPath, cNewStates)
                frontier.append(new_path)
    return ([],cNewPath, cNewStates)
```

We'll do the same for our ```mcost``` function, but we are storing ```f```. The goal is to compute how long it would take with h1 to find problem's 8 solution :

In [3]:
import psutil
from collections import deque
from state_search_thin import path_length, queue_update

def mcost2(model, maxdepth, memory=True):
        """Minimal-cost search, with counters per f layer."""
        (start, goal, successors, cost) = model
        cExpanded, cNewPath, cNewStates = {}, {}, {}
        if not cost:
            cost = path_length
        if goal(start):
            return ([start], {}, {}, {})
        frontier = deque([[start]])
        explored = set([])
        leaves = {start: [start]}
        fCurrent = cost([start])
        tick = time.perf_counter()
        while frontier:
            path = frontier.popleft()
            f = cost(path)
            if f > fCurrent: # Layer fCurrent is complete
                print(f"f = {fCurrent} : {cExpanded.get(fCurrent, 0)} expanded, {cNewStates.get(fCurrent, 0)} new states, {cNewPath.get(fCurrent, 0)} new paths ({time.perf_counter() - tick:.2f} s, {psutil.Process().memory_info().rss / 1024**2:.0f} MB)")
                fCurrent = f
            cExpanded[f] = cExpanded.get(f, 0) + 1
            state = path[-1]
            leaves.pop(state)
            if goal(state):
                return (path, cExpanded, cNewStates, cNewPath)
            if memory:
                explored.add(state)
            if path_length(path) >= maxdepth:
                continue
            for action, new_state in successors(state):
                new_path = path + [action, new_state]
                l = cost(new_path)
                cNewStates[l] = cNewStates.get(l, 0) + 1
                known = path if memory == False else explored
                if new_state not in known:
                    curr = leaves.get(new_state, None)
                    if curr:
                        if l < cost(curr):
                            frontier.remove(curr)
                        else:
                            continue
                    cNewPath[l] = cNewPath.get(l, 0) + 1
                    queue_update(frontier, new_path, cost)
                    leaves[new_state] = new_path
        return ([], cExpanded, cNewStates, cNewPath)

In [4]:
problem = problems[8]
model = (problem.start, problem.is_goal, problem.go_next, problem.astar_cost)
print(f"h1(start) = {problem.h1(problem.start)}")
result = mcost2(model, len(problem.answer))

h1(start) = 56
f = 56 : 1 expanded, 0 new states, 0 new paths (0.00 s, 127 MB)
f = 58 : 4 expanded, 6 new states, 4 new paths (0.00 s, 127 MB)
f = 60 : 294 expanded, 305 new states, 294 new paths (0.01 s, 127 MB)
f = 62 : 3916 expanded, 4471 new states, 3916 new paths (0.21 s, 127 MB)
f = 64 : 47483 expanded, 55347 new states, 47489 new paths (2.59 s, 131 MB)
f = 66 : 512438 expanded, 611352 new states, 512716 new paths (89.62 s, 478 MB)


KeyboardInterrupt: 

### Results for problem 8

Run stopped manually during layer f = 68 (RAM was filling up). Memory is the current RSS of the process (`psutil`), measured from a fresh kernel. Output:

| f layer | expanded | growth vs previous layer | elapsed time | RAM |
|---|---|---|---|---|
| 56 | 1 | | 0.00 s | 127 MB |
| 58 | 4 | | 0.00 s | 127 MB |
| 60 | 294 | x73 | 0.01 s | 127 MB |
| 62 | 3 916 | x13.3 | 0.21 s | 127 MB |
| 64 | 47 483 | x12.1 | 2.59 s | 131 MB |
| 66 | 512 438 | x10.8 | 89.62 s | 478 MB |

**What we learn:**
- f only goes up by 2 (each move changes h1 by +1 or -1 and the cost by +1), so the layers are 56, 58, 60, ...
- Layer 66 was fully expanded without reaching the goal, so the optimal solution has **at least 68 moves**.
- Each layer costs about **11 times** more than the previous one: this is the effective branching factor b* of A* with h1.
- **Speed:** layer 66 took 87 s for 512 438 paths, so about **5 900 paths/s**. It keeps dropping as the frontier grows (about 20 000 paths/s at f = 64), because `queue_update` and `frontier.remove` are O(n) on a deque.
- **Memory:** the search itself uses 478 - 127 = 351 MB for 564 136 expanded paths in total, so about **650 bytes per expanded path**. This covers everything the search keeps: the frontier (including the f = 68 children already generated), `explored` and `leaves`.

**Extrapolation (b* = 11, 5 900 paths/s, 650 bytes per path):**

| solution length | paths to expand (cumulated) | time (optimistic) | memory |
|---|---|---|---|
| 68 | about 6.2 million | about 17 min | about 4 GB |
| 70 | about 68 million | about 3.2 h | about 45 GB |
| 72 | about 750 million | about 1.5 days | about 490 GB |
| 74 | about 8 billion | about 16 days | about 5 TB |
| 80 | about 10<sup>13</sup> | about 60 years | about 7 PB |

The time column is optimistic: the speed keeps decreasing as the frontier grows.

**Conclusion:** if the optimal solution is exactly 68 moves long, A* with h1 would find it in about 17 minutes with about 4 GB of RAM. For any longer solution, memory runs out first: layer 70 already needs about 45 GB, far more than this 16 GB machine. A stronger heuristic (smaller b*) or a memory-bounded algorithm such as IDA* is needed.

(AI generated cell for clarity)